# 01 · Veri Keşfi (EDA)

**Amaç:** Model kurmadan önce veriyi tanımak ve ön işlemede vereceğimiz kararları **veriye dayandırmak.**

| Bölüm | Soru |
|---|---|
| **A. Veri setinin yapısı** | Elimizde ne var? Hangisi veri, hangisi metadata? |
| **B. Katalog analizi** | Sınıflar dengeli mi? Hangi yılları alalım? Sızıntı var mı? |
| **C. Radar görüntülerini tanıma** | Tornado radarda neye benziyor? Grafikler nasıl okunur? |
| **D. Ham veri analizi** | Hangi değişkeni, hangi tarama açısını, hangi zaman adımını kullanalım? Değer aralıkları ve ölçüm hassasiyeti ne? |

Colab'da CPU yeterli. C ve D bölümleri için 2017 arşivinden yaklaşık 1.500 örnek okunur (5–10 dakika).

In [ ]:
# --- Kurulum: hem Google Colab'da hem kendi bilgisayarında çalışır ---
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")                     # işlenmiş veri Drive'da saklanır
    REPO = Path("/content/tornado-cnn")
    if not REPO.exists():
        try:                                          # repo private ise: Colab Secrets'taki token
            token = userdata.get("GITHUB_TOKEN") + "@"
        except Exception:                             # repo public ise token gerekmez
            token = ""
        !git clone -q https://{token}github.com/baevv/tornado-cnn.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip install -q h5netcdf h5py
    os.chdir(REPO)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(REPO))
print("Colab" if IN_COLAB else "Yerel", "| repo:", REPO)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import paths, tornet, viz, eda

cfg = paths.load_config()
DATA = paths.data_dir()
print("veri klasörü:", DATA)
print("ayarlar:", cfg)

## A. Veri setinin yapısı

### A1. Veri iki parçadan oluşuyor

```
catalog.csv  ──── filename sütunu ────>  .nc dosyası
(metadata: her örnek hakkında bilgi)      (ham radar verisi + dosya içi metadata)
203.133 satır, 38 MB                       her örnek bir dosya, toplam ~170 GB
```

* **Katalog** her örneğin ne zaman, nerede, hangi fırtınadan alındığını ve etiketini söyler. Radar verisi içermez.
* **`.nc` dosyası** radar ölçümlerinin kendisidir. Modelin göreceği veri buradan gelir.

### A2. Katalog sütunları: hangisi ne işe yarıyor?

Katalog modelin girdisi **değildir**. Örnekleri düzenlemek, etiketlemek ve bölmek için kullanılır.

| Sütun | Anlamı | Kullanım | Neden? |
|---|---|---|---|
| `category` | `TOR` / `WRN` / `NUL` | ✅ **Etiket** | TOR = 1, diğerleri = 0 |
| `type` | `train` / `test` | ✅ **Bölme** | Veri setinin hazır test kümesi |
| `episode_id` | Fırtına olayı | ✅ **Bölme** | Aynı fırtınanın örnekleri eğitim ve doğrulamaya bölünmesin (sızıntı) |
| `ef_number` | Tornado şiddeti (EF0–EF4; tornado yoksa −1) | ✅ Sadece analiz | Eğitime girmez; sonra "güçlü tornadolar daha mı kolay bulunuyor?" diye bakılır |
| `filename` | `.nc` dosyasının adı | ✅ Eşleştirme | Katalog satırını radar dosyasına bağlar |
| `start_time`, `end_time` | Örneğin kapsadığı zaman aralığı | ✅ Yıl seçimi | Yıl buradan çıkarılır |
| `event_id` | Tek bir tornado / uyarı kaydı | ❌ | Bölme için `episode_id` daha kapsayıcı |
| `lat`, `lon`, `site_name` | Fırtınanın konumu, radar istasyonu | ❌ | Model konumdan kısayol öğrenmesin ("Oklahoma'daysa tornado") |
| `az_min`, `az_max`, `rng_min`, `rng_max` | Görüntünün radara göre açı ve mesafe sınırları | ❌ | Bu bilgi zaten `.nc` dosyasında var |

Biz iki sütun ekliyoruz: `year` (yıl) ve `label` (TOR = 1, WRN/NUL = 0).

| `category` | anlamı | `label` |
|---|---|---|
| `TOR` | doğrulanmış tornado | **1** |
| `WRN` | tornado uyarısı verilmiş ama tornado çıkmamış (**zor negatif**) | 0 |
| `NUL` | rastgele seçilmiş fırtına, tornado yok | 0 |

In [ ]:
cat = tornet.get_catalog(DATA)
print(f"{len(cat):,} örnek")
used_cols = ["filename", "type", "category", "label", "ef_number", "episode_id", "year"]
cat[used_cols].head()

### A3. Bir `.nc` dosyasının içi: hangisi veri, hangisi metadata?

| İçerik | Tür | Kullanım |
|---|---|---|
| `DBZ`, `VEL`, `KDP`, `RHOHV`, `ZDR`, `WIDTH` | **Ham radar verisi** (6 değişken) | Hangilerini kullanacağımıza **D bölümünde** karar veriyoruz |
| `frame_labels` | Her zaman adımının etiketi | ✅ Etiketin kaynağı (son kare) |
| `time`, `azimuth`, `range`, `elevation` | Koordinatlar (metadata) | Sadece grafik çizmek için |
| `nyquist_velocity` | Radarın ölçebildiği en yüksek hız | ❌ Ölçüm ayarı, fırtına hakkında bilgi değil |
| `range_folded_mask` | Ölçümün belirsiz olduğu pikseller | ❌ Bu pikseller zaten boş (NaN) olarak geliyor |

**6 radar değişkeninin anlamı:**

| Değişken | Ölçtüğü şey | Tornado ile ilişkisi |
|---|---|---|
| **DBZ** (yansıtıcılık) | Havada ne kadar yağmur/dolu var | Fırtınanın şekli; tornadolu fırtınalarda bazen **kanca** şekli |
| **VEL** (radyal hız) | Rüzgarın radara doğru (−) ya da radardan uzağa (+) hızı | Tornado = yan yana yaklaşan ve uzaklaşan rüzgar (**hız çifti**) |
| **WIDTH** (hız dağınıklığı) | Bir pikseldeki hızların ne kadar dağınık olduğu | Türbülans ve dönme bu değeri artırır |
| **KDP** | Yağışın yoğunluğu (çift kutuplu ölçüm) | Yoğun yağış, dolu |
| **RHOHV** | Pikseldeki parçacıkların ne kadar benzer olduğu | Tornadonun kaldırdığı **enkaz** düşük değer verir |
| **ZDR** | Parçacıkların yatay/dikey şekli | Enkaz değeri 0'a yaklaştırır |

Her değişkenin boyutu `(time=4, azimuth=120, range=240, sweep=2)`: 5 dakika arayla **4 zaman adımı**, 0,5°'lik **120 açı** hücresi (60°), 250 m'lik **240 mesafe** hücresi (60 km) ve 0,5° ile 0,9°'lik **2 tarama açısı**.

In [ ]:
first = tornet.sample_from_tar(tornet.tar_url(2017), per_category={"TOR": 1})
ds0 = next(iter(first.values()))
pd.DataFrame(
    [{"değişken": k, "boyutlar": str(v.dims), "şekil": str(v.shape), "tür": str(v.dtype)} for k, v in ds0.data_vars.items()]
).set_index("değişken")

## B. Katalog analizi

### B1. Sınıf dağılımı ve dengesizlik

`tornado_%` her yılda tornado örneklerinin oranı. %4–10 arası: veri **çok dengesiz.** Model her şeye "tornado yok" dese bile %90'dan fazla doğruluk alır. Bu yüzden doğruluk (accuracy) tek başına yanıltıcı olur; ROC-AUC, PR-AUC ve F1 kullanacağız.

In [ ]:
per_year = pd.crosstab(cat["year"], cat["category"])
per_year["total"] = per_year.sum(axis=1)
per_year["tornado_%"] = (100 * per_year["TOR"] / per_year["total"]).round(1)
sizes_gb = {2013: 3.2, 2014: 15.1, 2015: 17.4, 2016: 16.0, 2017: 15.1,
            2018: 12.5, 2019: 18.2, 2020: 17.0, 2021: 18.0, 2022: 19.0}   # Zenodo'daki arşiv boyutu
per_year["archive_GB"] = pd.Series(sizes_gb)
per_year["TOR_per_GB"] = (per_year["TOR"] / per_year["archive_GB"]).round(0)
per_year

### B2. Yıl seçimi

Disk ve süre kısıtı yüzünden 10 yılın hepsini kullanamıyoruz. En değerli ve en nadir örnek tornado olduğu için yılları **indirilen her GB başına kaç tornado örneği verdiğine** (`TOR_per_GB`) göre seçtik: en verimli iki yıl **2019** ve **2017**, ek veri için **2015**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5), layout="constrained")
colors = ["C1" if y in cfg["years"] else "C7" for y in per_year.index]
per_year["TOR"].plot.bar(ax=axes[0], color=colors)
axes[0].set_title("Tornado örneği sayısı")
per_year["TOR_per_GB"].plot.bar(ax=axes[1], color=colors)
axes[1].set_title("İndirilen GB başına tornado örneği")
for ax in axes:
    ax.set_xlabel("yıl")
fig.suptitle("Turuncu = projede kullanılan yıllar")
plt.show()

sel = cat[cat["year"].isin(cfg["years"])]
print("Yıllar:", cfg["years"], f"| {len(sel):,} örnek | {sel['label'].sum():,} tornado (%{100*sel['label'].mean():.1f})")
pd.crosstab(sel["type"], sel["category"], margins=True)

### B3. Tornado şiddeti (EF ölçeği)

Tornadoların çoğu zayıf (EF0–EF1). Güçlü tornadolar (EF3–EF4) çok nadir; 2017'de hiç EF4 yok.

In [ ]:
tor = cat[cat["category"] == "TOR"]
pd.crosstab(tor["year"], tor["ef_number"].astype(int), margins=True)

### B4. Eğitim/test ayrımı ve sızıntı kontrolü

`type` sütunundaki ayrımı veri setini hazırlayan MIT ekibi yapmış. Aynı fırtına olayı (`episode_id`) iki tarafta da varsa model sınavda daha önce gördüğü bir fırtınayla karşılaşır (**veri sızıntısı**). Ortak fırtınalar eğitimden çıkarılacak.

In [ ]:
train_eps = set(cat.loc[cat["type"] == "train", "episode_id"])
test_eps = set(cat.loc[cat["type"] == "test", "episode_id"])
overlap = train_eps & test_eps
print(f"test fırtınası: {len(test_eps):,} | eğitimde de olan: {len(overlap)}")
print(f"bu fırtınalara ait eğitim örneği: {cat[(cat['type']=='train') & cat['episode_id'].isin(overlap)].shape[0]}")

## C. Radar görüntülerini tanıma

### C0. Ham veriden örnek okuma

2017 arşivinin başından **1.500 örnek** okuyoruz (arşivin tamamı 15 GB; bu kısım ~1 GB). Bu örnekler C ve D bölümlerinde kullanılacak. Her örnek için D bölümündeki ölçüler de bu sırada hesaplanır.

Önce örneğin **temsil edici** olup olmadığına bakalım: TOR/WRN/NUL oranları 2017'nin tamamıyla benzer olmalı.

In [ ]:
scan = eda.scan_archive(tornet.tar_url(2017), cat, n_files=1500)
feat = scan["features"]

rep = pd.DataFrame({
    "okunan 1.500 örnek (%)": (100 * feat["category"].value_counts(normalize=True)).round(1),
    "2017'nin tamamı (%)": (100 * cat[cat["year"] == 2017]["category"].value_counts(normalize=True)).round(1),
})
rep

### C1. Grafik nasıl okunur?

**Eksenler:** Radar (0, 0) noktasında, çoğu zaman grafiğin dışında kalır. Yatay eksen radarın kaç km doğusunda, dikey eksen kaç km kuzeyinde olduğunu gösterir. Görüntü **yelpaze dilimi** şeklindedir, çünkü radar dönerek tarar ve bize fırtınanın çevresindeki 60° × 60 km'lik bir dilim verilmiştir.

**DBZ: yağış yoğunluğu**

| Renk | Değer | Anlamı |
|---|---|---|
| Mavi | 0–15 dBZ | Çok hafif yağış |
| Yeşil | 20–30 dBZ | Hafif-orta yağmur |
| Sarı-turuncu | 35–45 dBZ | Kuvvetli yağmur |
| Kırmızı | 50+ dBZ | Çok şiddetli yağmur, dolu |

**VEL: rüzgarın radara göre hızı**

| Renk | Anlamı |
|---|---|
| Kırmızı | Rüzgar radardan **uzaklaşıyor** |
| Mavi | Rüzgar radara **yaklaşıyor** |
| Beyaz | Hız sıfıra yakın |

**WIDTH: hız dağınıklığı**

| Renk | Anlamı |
|---|---|
| Koyu (siyah-mor) | Sakin, düzenli hava |
| Açık (turuncu-sarı) | Türbülans, hızlar çok dağınık |

**Tornadonun imzası:** Tornado dönen bir hava sütunudur. Bir yanı radara yaklaşır, öbür yanı uzaklaşır. Bu yüzden VEL'de **güçlü mavi ile güçlü kırmızı yan yana** durur (**hız çifti**). Aynı yerde WIDTH de yükselir.

```
   🟦🟦🟥🟥   ← yan yana yaklaşan ve uzaklaşan rüzgar = dönme
   🟦🟦🟥🟥
```

**Siyah daire:** Kod, görüntünün merkezinde VEL'deki en güçlü hız çiftini bulup üç grafikte de aynı yeri daire içine alır.

Beyaz (boş) alanlar ölçüm olmayan yerlerdir: orada yağış ya da yansıma yoktur.

### C2. Güçlü bir tornado (EF2 ve üstü)

Okunan tornado örnekleri arasından EF2 ve üstü olan ve dönmesi en güçlü olanı seçiyoruz. Hız çifti bu örneklerde net görünür.

In [ ]:
meta = cat.set_index("filename")
tor_raw = {n: b for n, b in scan["raw"].items() if "/TOR_" in n}
tor_info = pd.DataFrame({
    "ef": [meta.at[n, "ef_number"] for n in tor_raw],
    "dönme_gücü": [feat.set_index("filename").at[n, "VEL_s0"] for n in tor_raw],
}, index=list(tor_raw))

strong_name = tor_info[tor_info["ef"] >= 2].sort_values("dönme_gücü", ascending=False).index[0]
strong = tornet.open_sample(tor_raw[strong_name])
viz.plot_sample(strong)
plt.show()

### C3. Zayıf tornado, uyarı ve rastgele fırtına

Karşılaştırma için:

* **Zayıf tornado (EF0):** Dönme o kadar küçük ki çoğu zaman bu çözünürlükte görünmez. Model için en zor örnekler bunlar.
* **Uyarı (WRN):** Tehlikeli görünen ama tornado çıkmayan fırtına.
* **Rastgele fırtına (NUL):** Çoğunlukla sakin.

Dairenin bu örneklerde de çizildiğine dikkat: kod her zaman "en güçlü dönme"yi işaretler. Tornado olmayan örneklerde orada belirgin bir hız çifti yoktur.

In [ ]:
weak_name = tor_info[tor_info["ef"] == 0].sort_values("dönme_gücü").index[len(tor_info[tor_info["ef"] == 0]) // 2]
wrn_name = next(n for n in scan["raw"] if "/WRN_" in n)
nul_name = next(n for n in scan["raw"] if "/NUL_" in n)
for name in (weak_name, wrn_name):
    raw = scan["raw"][name]
    viz.plot_sample(tornet.open_sample(raw))
    plt.show()
viz.plot_sample(tornet.open_sample(scan["raw"][nul_name]))
plt.show()

### C4. Güçlü tornadonun 4 zaman adımı

`frame_labels` tornadonun yerde olduğu kareleri işaretler (başlıklardaki `etiket`). Örneğin etiketi **son** karenin etiketidir.

In [ ]:
viz.plot_frames(strong, "VEL")
plt.show()

## D. Ham veri analizi: kararların kanıtı

Bu bölümde C0'da okunan 1.500 örneğin hepsi kullanılıyor. Her soru bir ön işleme kararına karşılık geliyor.

### D1. Boş piksel oranı

Radar her yeri ölçemez: yağış yoksa yansıma da yoktur. Tablo, her değişkende piksellerin yüzde kaçının boş (NaN) olduğunu gösteriyor (en alçak tarama, son kare).

In [ ]:
nan_cols = [f"{v}_s0_nan" for v in eda.VARIABLES]
nan_table = (100 * feat.groupby("category")[nan_cols].mean()).round(1)
nan_table.columns = eda.VARIABLES
nan_table

* Boşluklar her değişkende var, bu yüzden ön işlemede boş pikselleri **doldurmak zorundayız** (model "boş" değeri anlayamaz).
* VEL ve WIDTH'in boşlukları aynı, çünkü ikisi de aynı Doppler ölçümünden gelir.
* Tornado örneklerinde boşluk daha az: tornadolu fırtınalar daha geniş ve yoğun.

### D2. Değer dağılımları ve ölçekleme aralıkları

Modele vermeden önce her kanalı sabit bir aralıkla [-1, 1]'e ölçekleyeceğiz. Aralık verinin neredeyse tamamını kapsamalı; dışında kalan değerler aralığın sınırına kırpılır. Kesikli çizgiler seçtiğimiz aralığı gösteriyor.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6), layout="constrained")
for ax, v in zip(axes.flat, eda.VARIABLES):
    bins = eda.HIST_BINS[v]
    ax.bar(bins[:-1], scan["hists"][v], width=np.diff(bins), align="edge", color="C0")
    ax.set_yscale("log")
    ax.set_title(v)
    if v in tornet.CHANNEL_RANGES and v in cfg["variables"]:
        for b in tornet.CHANNEL_RANGES[v]:
            ax.axvline(b, color="red", ls="--")
fig.suptitle("Piksel değerlerinin dağılımı (logaritmik ölçek). Kırmızı = kullanılan kanalların ölçekleme aralığı")
plt.show()

pd.DataFrame({
    v: {"aralık": str(tornet.CHANNEL_RANGES[v]),
        "aralık dışı piksel (%)": round(100 * scan["outside"][v][0] / scan["outside"][v][1], 2)}
    for v in scan["outside"]
}).T

**WIDTH aralığı neden 0–16?** MIT'nin referans kodu WIDTH için 0–9 kullanıyor. Aşağıdaki hücre, 0–9 seçilseydi kaç pikselin kırpılacağını hesaplıyor. Tornadoda tam da yüksek WIDTH değerlerini beklediğimiz için bu pikselleri kaybetmek istemiyoruz; aralığı verimize göre genişlettik.

In [ ]:
bins, counts = eda.HIST_BINS["WIDTH"], scan["hists"]["WIDTH"]
for hi in (9, 16):
    print(f"WIDTH 0–{hi} aralığının dışında kalan piksel: %{100 * counts[bins[:-1] >= hi].sum() / counts.sum():.2f}")

### D3. Hangi değişken tornadoyu ayırt ediyor?

Her örnek için her değişkenden **tek bir sayı** hesaplıyoruz. Görüntünün merkezinde (15° × 15 km), gürültü temizlendikten sonra (ayrıntılar `src/eda.py`'de):

| Değişken | Ölçü | Neden bu ölçü? |
|---|---|---|
| DBZ | en yüksek değerler (99. yüzdelik) | Güçlü fırtına = yüksek yansıtıcılık |
| VEL | **dönme gücü**: küçük bir pencerede en büyük hız farkı | Tornado = hız çifti |
| WIDTH, KDP | en yüksek değerler (99. yüzdelik) | Türbülans / yoğun yağış |
| RHOHV, ZDR | en düşük değerler (1. yüzdelik) | Enkaz bu değerleri düşürür |

Sonra her ölçünün tornadoyu ne kadar ayırt ettiğini **AUC** ile puanlıyoruz:

* **0,5:** Ölçü tornadoyu hiç ayırt etmiyor (yazı tura).
* **1,0:** Mükemmel ayırt ediyor.
* **0,5'in altı:** Ters yönde ayırt ediyor (tornadoda daha düşük). Bu yüzden "ayırt etme gücü" = max(AUC, 1 − AUC) da veriliyor.

İki soru soruyoruz: **"Tornado mu, değil mi?"** (TOR – WRN+NUL) ve daha zor olanı **"Tornado mu, uyarı mı?"** (TOR – WRN).

In [ ]:
cols0 = [f"{v}_s0" for v in eda.VARIABLES]
fig, axes = plt.subplots(1, 6, figsize=(17, 3.6), layout="constrained")
order = ["TOR", "WRN", "NUL"]
for ax, c, v in zip(axes, cols0, eda.VARIABLES):
    data = [feat.loc[feat["category"] == k, c].dropna() for k in order]
    ax.boxplot(data, showfliers=False)
    ax.set_xticks([1, 2, 3], order)
    ax.set_title(v)
fig.suptitle("Her değişkenin özet ölçüsü, kategoriye göre (kutu = ortadaki %50)")
plt.show()

auc_all = eda.separability(feat, cols0)
auc_wrn = eda.separability(feat, cols0, negative=("WRN",))
auc_table = pd.DataFrame({
    "Tornado mu? (AUC)": auc_all["AUC"],
    "ayırt etme gücü": auc_all["ayırt etme gücü"],
    "Tornado mu, uyarı mı? (AUC)": auc_wrn["AUC"],
    "ayırt etme gücü ": auc_wrn["ayırt etme gücü"],
})
auc_table.index = eda.VARIABLES
auc_table.sort_values("ayırt etme gücü", ascending=False)

### Kanal seçimi

Tablo ve kutu grafikleri şunu gösteriyor (bizim çalıştırmamızdaki değerlerle):

* **VEL açık ara en iyi** (≈0,76). En zor soruda, tornado ile uyarıyı ayırmada da en iyisi (≈0,69). Tornadonun ana imzası olan hız çifti bu kanalda. → **Seçildi.**
* **WIDTH ikinci** (≈0,69). Dönme ve türbülans hızların dağınıklığını artırıyor. → **Seçildi.**
* **DBZ** "fırtına var mı?" sorusunda bir miktar işe yarıyor (≈0,62) ama **tornadoyu uyarıdan ayıramıyor** (≈0,48). Ancak bu ölçü görüntüyü tek bir sayıya indiriyor; DBZ'nin asıl bilgisi **şekilde** (kanca), ve onu tek sayı ölçemez, CNN görebilir. Meteorolojide de tornado tespitinin standart kanalı. → **Seçildi.**
* **KDP** DBZ ile aynı türden bilgi taşıyor (yağış yoğunluğu). **RHOHV** ve **ZDR**'nin ayırt ediciliği düşük; bunların asıl sinyali güçlü tornadoların kaldırdığı enkaz, ve verimizin çoğu zayıf tornado. → **Alınmadı.**

**Sonuç:** Model **DBZ + VEL + WIDTH** (3 kanal) ile eğitilecek. Kanalların gerçek katkısı ayrıca bir eğitim deneyiyle (06 defteri) sınanacak.

**Ölçüleri neden merkezde hesapladık?** Aşağıdaki hücre, görüntünün tamamında en güçlü dönmenin nerede olduğuna bakıyor. Tornado örneklerinde en güçlü dönme çok daha sık merkezde: veri seti, olayın konumu görüntünün ortasında olacak şekilde kesilmiş.

In [ ]:
center = (feat["rot_az"].between(45, 74)) & (feat["rot_rng"].between(90, 149))
(100 * center.groupby(feat["category"]).mean()).round(1).rename("en güçlü dönme merkezde (%)").to_frame()

### D4. Hangi tarama açısı?

Aynı ölçüler iki tarama açısında: 0,5° (yere en yakın) ve 0,9°. Radar ışını uzaklaştıkça yükseldiği için alçak tarama yere, yani tornadoya daha yakın bakar.

In [ ]:
cols1 = [f"{v}_s1" for v in eda.VARIABLES]
sweeps = pd.DataFrame({
    "0,5° (AUC)": eda.separability(feat, cols0)["ayırt etme gücü"].values,
    "0,9° (AUC)": eda.separability(feat, cols1)["ayırt etme gücü"].values,
}, index=eda.VARIABLES)
sweeps["fark"] = (sweeps["0,5° (AUC)"] - sweeps["0,9° (AUC)"]).round(3)
sweeps

İki tarama arasındaki farklar küçük. En önemli kanal olan VEL'de alçak tarama biraz daha iyi. Fiziksel gerekçe de alçak taramadan yana. İkisini birden almak veriyi iki katına çıkarırdı; bu yüzden **en alçak tarama (0,5°)** seçildi.

### D5. Hangi zaman adımı?

Her örnekte 4 kare var. Etiketin hangi kareden geldiğini ve tornadonun hangi karede başladığını kontrol edelim.

In [ ]:
t = feat[feat["category"] == "TOR"]
print("TOR örneklerinde son karenin etiketi 1 mi?        ", (t["last_label"] == 1).all())
print("TOR olmayan örneklerde bütün kareler 0 mı?        ", (feat.loc[feat["category"] != "TOR", "n_tor_frames"] == 0).all())
frames = t["first_tor_frame"].value_counts().sort_index()
frames.index = [f"{i + 1}. kare" for i in frames.index]
frames.rename("tornadonun başladığı kare (örnek sayısı)").to_frame()

* Etiket **son karenin** etiketi: bütün TOR örneklerinde son kare 1, diğer örneklerde bütün kareler 0.
* Tornadoların çoğu ancak son karelerde başlıyor; önceki karelerde henüz tornado yok.

"Şu an tornado var mı?" sorusuna cevap aradığımız için **son kare** seçildi. Önceki kareler ileride (RNN/LSTM haftaları) tornadonun oluşumunu modellemek için kullanılabilir.

### D6. Ölçüm hassasiyeti

Ön işlemede veriyi küçültmek için her pikseli 0–255 arası tam sayıya (`uint8`) çevireceğiz. Bu, değerleri 254 adıma yuvarlamak demek. Bilgi kaybedip kaybetmediğimizi anlamak için radarın veriyi hangi hassasiyetle kaydettiğine bakalım.

In [ ]:
rows = []
for v in cfg["variables"]:
    lo, hi = tornet.CHANNEL_RANGES[v]
    rows.append({
        "kanal": v,
        "radarın kaydettiği en sık adım": scan["steps"][v].index[0],
        "uint8 adımı": round((hi - lo) / 254, 3),
        "en büyük yuvarlama hatası": round((hi - lo) / 254 / 2, 3),
    })
pd.DataFrame(rows).set_index("kanal")

* **DBZ** ve **WIDTH** radarda 0,5'lik adımlarla kayıtlı; `uint8` adımımız daha ince, yani **bilgi kaybı yok.**
* **VEL**'de bazı değerler daha ince adımlarla kayıtlı. En büyük yuvarlama hatası ≈0,24 m/s; tornadodaki 20–60 m/s'lik hız farkları yanında önemsiz.

Bu kararlar `02_preprocess.ipynb` defterinde uygulanıyor.